# HuggingFace Transformers Experiments

This notebook demonstrates various NLP tasks using HuggingFace Transformers library.

**Tasks covered:**
1. Text Classification with BERT
2. Named Entity Recognition
3. Text Generation with GPT-2
4. Question Answering

In [ ]:
# 设置镜像和代理 (加速 HuggingFace 下载)
import os

# 使用 HuggingFace 镜像 (国内用户)
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

# 同时启用代理 (某些模型可能需要)
# os.environ['HTTP_PROXY'] = 'http://127.0.0.1:7890'
# os.environ['HTTPS_PROXY'] = 'http://127.0.0.1:7890'

print("镜像和代理已设置")
print("HF_ENDPOINT:", os.environ.get('HF_ENDPOINT'))
print("HTTP_PROXY:", os.environ.get('HTTP_PROXY'))

镜像和代理已设置
HF_ENDPOINT: https://hf-mirror.com
HTTP_PROXY: http://127.0.0.1:7890


In [12]:
import torch
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    AutoModelForTokenClassification,
    AutoModelForCausalLM,
    AutoModelForQuestionAnswering,
    AutoModelForSeq2SeqLM,
    pipeline
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Transformers version: {__import__('transformers').__version__}")

Using device: cuda
Transformers version: 5.5.1


## 1. Text Classification with BERT

Classify text into categories using pre-trained BERT model fine-tuned for sentiment analysis.

In [13]:
# Use a pre-trained sentiment analysis model
classifier = pipeline("sentiment-analysis", device=0 if torch.cuda.is_available() else -1)

texts = [
    "I love this movie! It's absolutely fantastic!",
    "This product is terrible. Waste of money.",
    "The service was okay, nothing special.",
    "Amazing experience, highly recommended!"
]

results = classifier(texts)

print("Sentiment Analysis Results:")
print("-" * 50)
for text, result in zip(texts, results):
    print(f"Text: {text}")
    print(f"Label: {result['label']}, Score: {result['score']:.4f}")
    print()

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Sentiment Analysis Results:
--------------------------------------------------
Text: I love this movie! It's absolutely fantastic!
Label: POSITIVE, Score: 0.9999

Text: This product is terrible. Waste of money.
Label: NEGATIVE, Score: 0.9998

Text: The service was okay, nothing special.
Label: NEGATIVE, Score: 0.9862

Text: Amazing experience, highly recommended!
Label: POSITIVE, Score: 0.9999



## 2. Named Entity Recognition (NER)

Extract named entities (persons, organizations, locations) from text.

In [14]:
# NER pipeline - 使用 aggregation_strategy 替代 grouped_entities
ner_pipeline = pipeline("ner", aggregation_strategy="simple", device=0 if torch.cuda.is_available() else -1)

text = """Apple Inc. is headquartered in Cupertino, California. 
Tim Cook has been the CEO since 2011. The company was founded by Steve Jobs, 
Steve Wozniak, and Ronald Wayne in April 1976."""

entities = ner_pipeline(text)

print("Named Entity Recognition Results:")
print("-" * 50)
for entity in entities:
    print(f"Entity: {entity['word']}")
    print(f"  Type: {entity['entity_group']}")
    print(f"  Score: {entity['score']:.4f}")
    print()

No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Named Entity Recognition Results:
--------------------------------------------------
Entity: Apple Inc
  Type: ORG
  Score: 0.9995

Entity: Cupertino
  Type: LOC
  Score: 0.9666

Entity: California
  Type: LOC
  Score: 0.9986

Entity: Tim Cook
  Type: PER
  Score: 0.9997

Entity: Steve Jobs
  Type: PER
  Score: 0.9937

Entity: Steve Wozniak
  Type: PER
  Score: 0.9304

Entity: Ronald Wayne
  Type: PER
  Score: 0.9996



## 3. Text Generation with GPT-2

Generate text using GPT-2 language model.

In [15]:
generator = pipeline("text-generation", model="gpt2", device=0 if torch.cuda.is_available() else -1)

prompts = [
    "Once upon a time in a distant land,",
    "The future of artificial intelligence is",
    "In the year 2050, humans will"
]

print("Text Generation Results:")
print("=" * 60)

for prompt in prompts:
    print(f"\nPrompt: {prompt}")
    output = generator(prompt, max_length=80, num_return_sequences=1, temperature=0.8)
    print(f"Generated: {output[0]['generated_text']}")
    print("-" * 60)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=80) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Text Generation Results:

Prompt: Once upon a time in a distant land,


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=80) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generated: Once upon a time in a distant land, an army of noblemen were ordered to follow the king to his bedside. They were forced to wait for the king to make his appearance. But when the king emerged, all of them were terrified and went back to their own home to watch his performance."

As for the other two generals, the king himself was a poor man. He had to rely on his troops and his own luck for his survival. To do so, he relied on the people of his country to help him.

Although the soldiers seemed to have a lot of strength, they also didn't have a great many skills. With the exception of the sword, all the army's equipment consisted of spears, axes, spears, bows, javelins, and weapons.

At this time, the king had a man who was also quite strong. It was a man named Caius.

"What is this?"

"The people of the city of Zangzhou have a bad reputation. Their leaders are a little bit too arrogant, but their people are also very honest and strong. When they see something, they always l

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=80) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generated: The future of artificial intelligence is increasingly uncertain. But it's just as interesting to think of a future in which artificial intelligence is an extension of human intelligence.

What is AI?

Artificial intelligence is a scientific discipline based on the understanding of the human cognitive system rather than the machine cognitive system. Artificial intelligence is a new discipline under the supervision of the artificial intelligence community, which is a collection of experts who work in various fields. The AI community's main focus is on natural language processing and machine learning, which are more common in advanced fields such as biology, psychology, and robotics.

"Human-machine co-maintaining of the language processing and machine learning processes will need to be taken into account," said M.A. Leff, Chief Scientific Officer of AI Research. "We are in a deep state of mind where we cannot simply rely on 'human' for information."

According to a paper publi

## 4. Question Answering

Answer questions based on a given context using BERT.

In [16]:
# Question Answering - 直接使用模型 (新版 transformers 不支持 question-answering pipeline)
from transformers import AutoModelForQuestionAnswering, AutoTokenizer

qa_model_name = "distilbert-base-cased-distilled-squad"
qa_tokenizer = AutoTokenizer.from_pretrained(qa_model_name)
qa_model = AutoModelForQuestionAnswering.from_pretrained(qa_model_name).to(device)

context = """The Transformer architecture was introduced in the paper "Attention Is All You Need" 
by Google researchers in 2017. It revolutionized natural language processing by using 
self-attention mechanisms instead of recurrent networks. BERT (Bidirectional Encoder 
Representations from Transformers) was released by Google in 2018 and became one of the 
most influential models in NLP. GPT (Generative Pre-trained Transformer) is another 
important model family developed by OpenAI."""

questions = [
    "When was the Transformer architecture introduced?",
    "What does BERT stand for?",
    "Who developed GPT?",
    "What mechanism does Transformer use?"
]

print("Question Answering Results:")
print("=" * 60)
print(f"Context: {context}\n")

qa_model.eval()
for question in questions:
    inputs = qa_tokenizer(question, context, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = qa_model(**inputs)
    
    answer_start = torch.argmax(outputs.start_logits)
    answer_end = torch.argmax(outputs.end_logits) + 1
    answer = qa_tokenizer.convert_tokens_to_string(
        qa_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0][answer_start:answer_end])
    )
    score = torch.max(torch.softmax(outputs.start_logits, dim=1)[0, answer_start],
                      torch.softmax(outputs.end_logits, dim=1)[0, answer_end-1]).item()
    
    print(f"Q: {question}")
    print(f"A: {answer} (score: {score:.4f})")
    print()

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Question Answering Results:
Context: The Transformer architecture was introduced in the paper "Attention Is All You Need" 
by Google researchers in 2017. It revolutionized natural language processing by using 
self-attention mechanisms instead of recurrent networks. BERT (Bidirectional Encoder 
Representations from Transformers) was released by Google in 2018 and became one of the 
most influential models in NLP. GPT (Generative Pre-trained Transformer) is another 
important model family developed by OpenAI.

Q: When was the Transformer architecture introduced?
A: 2017 (score: 0.9889)

Q: What does BERT stand for?
A: Bidirectional Encoder Representations from Transformers (score: 0.9309)

Q: Who developed GPT?
A: OpenAI (score: 0.9957)

Q: What mechanism does Transformer use?
A: self - attention mechanisms (score: 0.9639)



## 5. Text Summarization

Summarize long texts into shorter versions.

In [17]:
# Text Summarization - 直接使用模型
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

sum_model_name = "facebook/bart-large-cnn"
sum_tokenizer = AutoTokenizer.from_pretrained(sum_model_name)
sum_model = AutoModelForSeq2SeqLM.from_pretrained(sum_model_name).to(device)

long_text = """
Natural language processing (NLP) is a subfield of linguistics, computer science, and artificial intelligence 
concerned with the interactions between computers and human language, in particular how to program computers 
to process and analyze large amounts of natural language data. The result is a computer capable of "understanding" 
the contents of documents, including the contextual nuances of the language within them. The technology can then 
accurately extract information and insights contained in the documents as well as categorize and organize the documents 
themselves. Challenges in natural language processing frequently involve speech recognition, natural language understanding, 
and natural-language generation. Modern NLP algorithms are based on machine learning, especially statistical machine learning 
and deep learning. The era of deep learning in NLP started around 2013-2014 with the introduction of word embeddings 
like Word2Vec and GloVe. Later, recurrent neural networks (RNNs) and Long Short-Term Memory (LSTM) networks became popular 
for sequence modeling tasks. The most significant breakthrough came in 2017 with the introduction of the Transformer 
architecture, which enabled parallel processing of sequences and led to models like BERT, GPT, and many others.
"""

# Tokenize and generate summary
inputs = sum_tokenizer(long_text, max_length=1024, truncation=True, return_tensors="pt").to(device)
summary_ids = sum_model.generate(inputs["input_ids"], max_length=80, min_length=30, num_beams=4, early_stopping=True)
summary = sum_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("Original Text Length:", len(long_text.split()))
print("\nSummary:")
print(summary)
print("\nSummary Length:", len(summary.split()))

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

Original Text Length: 179

Summary:
Natural language processing (NLP) is a subfield of linguistics, computer science, and artificial intelligence. Modern NLP algorithms are based on machine learning, especially statistical machine learning and deep learning. Challenges in natural language processing frequently involve speech recognition, natural language understanding, and natural-language generation.

Summary Length: 44


## 6. Translation

Translate text between languages using pre-trained models.

In [19]:
# English to French Translation - 使用较小的模型
# 如果下载失败，可以跳过这个任务
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

trans_model_name = "t5-small"  # 使用 t5-small，模型更小更容易下载

try:
    trans_tokenizer = AutoTokenizer.from_pretrained(trans_model_name)
    trans_model = AutoModelForSeq2SeqLM.from_pretrained(trans_model_name).to(device)

    english_texts = [
        "Hello, how are you today?",
        "Machine learning is transforming the world.",
        "The quick brown fox jumps over the lazy dog."
    ]

    print("English to French Translation (using T5-small):")
    print("-" * 50)

    for text in english_texts:
        # T5 需要添加任务前缀
        inputs = trans_tokenizer("translate English to French: " + text, return_tensors="pt").to(device)
        translated = trans_model.generate(**inputs, max_length=50)
        translation = trans_tokenizer.decode(translated[0], skip_special_tokens=True)
        print(f"EN: {text}")
        print(f"FR: {translation}")
        print()
except Exception as e:
    print(f"Translation model download failed: {e}")
    print("Skipping translation task. You can try again later with better network.")

model.safetensors:   4%|4         | 10.5M/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

English to French Translation (using T5-small):
--------------------------------------------------
EN: Hello, how are you today?
FR: Bonjour, comment êtes-vous aujourd'hui?

EN: Machine learning is transforming the world.
FR: L'apprentissage en machine transforme le monde.

EN: The quick brown fox jumps over the lazy dog.
FR: Le renard brun rapide saute sur le chien paresseux.



## 7. Zero-Shot Classification

Classify text into custom categories without fine-tuning.

In [22]:
# Zero-Shot Classification - 使用更小的模型
zero_shot = pipeline("zero-shot-classification", 
                    model="typeform/distilbert-base-uncased-mnli",
                    device=0 if torch.cuda.is_available() else -1)

text = "The new smartphone features a 108MP camera and 5G connectivity."
candidate_labels = ["technology", "sports", "politics", "entertainment", "health"]

result = zero_shot(text, candidate_labels)

print("Zero-Shot Classification:")
print("-" * 50)
print(f"Text: {text}")
print(f"\nLabel scores:")
for label, score in zip(result['labels'], result['scores']):
    print(f"  {label}: {score:.4f}")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Falling back to torch.float32 because loading with the original dtype failed on the target device.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Zero-Shot Classification:
--------------------------------------------------
Text: The new smartphone features a 108MP camera and 5G connectivity.

Label scores:
  technology: 0.4459
  entertainment: 0.2220
  health: 0.1989
  politics: 0.0720
  sports: 0.0612


## 8. Custom Dataset: AG News Classification

Use BERT to classify AG News dataset.

In [23]:
import pandas as pd

# Load AG News dataset
data_path = '/home/ubuntu/AI-For-Beginners/lessons/5-NLP/15-LanguageModeling/lab/data/ag_news_csv'

train_df = pd.read_csv(f'{data_path}/train.csv', header=None, names=['label', 'title', 'description'])
test_df = pd.read_csv(f'{data_path}/test.csv', header=None, names=['label', 'title', 'description'])

classes = ['World', 'Sports', 'Business', 'Sci/Tech']

print(f"Training samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")
print(f"\nClasses: {classes}")
print(f"\nSample data:")
print(train_df.head(3))

Training samples: 120000
Test samples: 7600

Classes: ['World', 'Sports', 'Business', 'Sci/Tech']

Sample data:
   label                                              title  \
0      3  Wall St. Bears Claw Back Into the Black (Reuters)   
1      3  Carlyle Looks Toward Commercial Aerospace (Reu...   
2      3    Oil and Economy Cloud Stocks' Outlook (Reuters)   

                                         description  
0  Reuters - Short-sellers, Wall Street's dwindli...  
1  Reuters - Private investment firm Carlyle Grou...  
2  Reuters - Soaring crude prices plus worries\ab...  


In [24]:
# Use zero-shot classification for AG News
test_samples = test_df.head(10)

print("AG News Classification using Zero-Shot:")
print("=" * 60)

for idx, row in test_samples.iterrows():
    text = row['title'] + " " + row['description']
    true_label = classes[row['label'] - 1]
    
    result = zero_shot(text[:512], classes)  # Truncate to avoid token limit
    predicted_label = result['labels'][0]
    
    print(f"True: {true_label} | Predicted: {predicted_label}")
    print(f"Text: {row['title'][:60]}...")
    print(f"Scores: {dict(zip(result['labels'][:2], [f'{s:.3f}' for s in result['scores'][:2]]))}")
    print()

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


AG News Classification using Zero-Shot:
True: Business | Predicted: Business
Text: Fears for T N pension after talks...
Scores: {'Business': '0.348', 'World': '0.303'}

True: Sci/Tech | Predicted: Sci/Tech
Text: The Race is On: Second Private Team Sets Launch Date for Hum...
Scores: {'Sci/Tech': '0.361', 'World': '0.240'}

True: Sci/Tech | Predicted: Sci/Tech
Text: Ky. Company Wins Grant to Study Peptides (AP)...
Scores: {'Sci/Tech': '0.353', 'World': '0.285'}

True: Sci/Tech | Predicted: World
Text: Prediction Unit Helps Forecast Wildfires (AP)...
Scores: {'World': '0.437', 'Sci/Tech': '0.271'}

True: Sci/Tech | Predicted: Sci/Tech
Text: Calif. Aims to Limit Farm-Related Smog (AP)...
Scores: {'Sci/Tech': '0.344', 'World': '0.237'}

True: Sci/Tech | Predicted: Sci/Tech
Text: Open Letter Against British Copyright Indoctrination in Scho...
Scores: {'Sci/Tech': '0.459', 'World': '0.251'}

True: Sci/Tech | Predicted: Sci/Tech
Text: Loosing the War on Terrorism...
Scores: {'Sci/Tech': '0.84

## Summary

| Task | Model | Use Case |
|------|-------|----------|
| Sentiment Analysis | `distilbert-base-uncased-finetuned-sst-2-english` | Product reviews, social media |
| NER | `dbmdz/bert-large-cased-finetuned-conll03-english` | Entity extraction |
| Text Generation | `gpt2` | Creative writing, completion |
| Question Answering | `distilbert-base-cased-distilled-squad` | Document QA |
| Summarization | `facebook/bart-large-cnn` | Document summarization |
| Translation | `t5-base` | Multi-language translation |
| Zero-Shot Classification | `facebook/bart-large-mnli` | Custom categories without training |

### Key Findings:
1. **Pre-trained models** provide excellent out-of-the-box performance
2. **Zero-shot classification** works surprisingly well for many tasks
3. **Fine-tuning** can further improve performance on specific domains
4. **Pipeline API** makes it easy to use transformers with minimal code